In [3]:
import os
import shutil
import numpy as np
import requests
import zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from PIL import Image
from io import BytesIO

# Step 1: Download dataset from GitHub
def download_and_extract_dataset():
    dataset_url = "https://github.com/chandrikadeb7/Face-Mask-Detection/archive/refs/heads/master.zip"
    zip_path = "Face-Mask-Detection.zip"
    extract_path = "Face-Mask-Detection-master"

    print("Downloading dataset...")
    response = requests.get(dataset_url)
    with open(zip_path, 'wb') as file:
        file.write(response.content)

    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()

    os.remove(zip_path)
    print("Dataset downloaded and extracted.")

# Download and extract the dataset
download_and_extract_dataset()

# Step 2: Split dataset into train and validation sets manually
def split_dataset(base_dir, train_dir, val_dir, split_ratio=0.8):
    # Create directories for train and validation data
    for category in ['with_mask', 'without_mask']:
        os.makedirs(os.path.join(train_dir, category), exist_ok=True)
        os.makedirs(os.path.join(val_dir, category), exist_ok=True)

        # Get list of images
        category_dir = os.path.join(base_dir, category)
        images = os.listdir(category_dir)
        np.random.shuffle(images)

        # Split into train and validation sets
        split_index = int(len(images) * split_ratio)
        train_images, val_images = images[:split_index], images[split_index:]

        # Copy images to train and validation folders
        for img in train_images:
            shutil.copy(os.path.join(category_dir, img), os.path.join(train_dir, category))
        for img in val_images:
            shutil.copy(os.path.join(category_dir, img), os.path.join(val_dir, category))
        print(f"Category '{category}' split into {len(train_images)} train and {len(val_images)} validation images.")

base_dir = "Face-Mask-Detection-master/dataset"
train_dir = "medical-mask-dataset/train"
val_dir = "medical-mask-dataset/validation"

split_dataset(base_dir, train_dir, val_dir)

# Step 3: Load the VGG16 pretrained model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(2, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Step 4: Load and preprocess the dataset
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_dataset = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

validation_dataset = test_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Step 5: Train the model
history = model.fit(
    train_dataset,
    epochs=5,
    validation_data=validation_dataset
)

# Step 6: Classify an image from a URL
def preprocess_image(image_url):
    response = requests.get(image_url)
    img = Image.open(BytesIO(response.content)).convert('RGB')
    img = img.resize((224, 224))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0
    return img_array

def test_image(image_url, model, class_names):
    img_array = preprocess_image(image_url)
    predictions = model.predict(img_array)
    predicted_class = np.argmax(predictions, axis=1)[0]
    confidence = predictions[0][predicted_class]
    print(f"Predicted class: {class_names[predicted_class]} with confidence {confidence:.2f}")



Extracting dataset...
Dataset downloaded and extracted.
Category 'with_mask' split into 1732 train and 433 validation images.
Category 'without_mask' split into 1544 train and 386 validation images.
Found 4053 images belonging to 2 classes.
Found 1986 images belonging to 2 classes.
Epoch 1/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 86s 640ms/step - accuracy: 0.8146 - loss: 0.4044 - val_accuracy: 0.9607 - val_loss: 0.1050
Epoch 2/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 123s 499ms/step - accuracy: 0.9366 - loss: 0.1589 - val_accuracy: 0.9723 - val_loss: 0.0745
Epoch 3/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 84s 509ms/step - accuracy: 0.9499 - loss: 0.1313 - val_accuracy: 0.9839 - val_loss: 0.0521
Epoch 4/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 66s 498ms/step - accuracy: 0.9610 - loss: 0.1113 - val_accuracy: 0.9804 - val_loss: 0.0463
Epoch 5/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 65s 492ms/step - accuracy: 0.9683 - loss: 0.1018 - val_accuracy: 0.9783 - val_loss: 0.0608
Enter image URL: https://www.google.com/url?sa=i&url=https%3A%2F%2

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x7e283cdc39c0>

In [4]:

# Example usage
image_url = input("Enter image URL: ")
test_image(image_url, model, list(train_dataset.class_indices.keys()))

Enter image URL: https://images.pexels.com/photos/2379004/pexels-photo-2379004.jpeg?auto=compress&cs=tinysrgb&dpr=1&w=500
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted class: without_mask with confidence 0.98


In [6]:

# Example usage
image_url = input("Enter image URL: ")
test_image(image_url, model, list(train_dataset.class_indices.keys()))

Enter image URL: https://media.istockphoto.com/id/1311420208/photo/woman-in-grey-blouse-wearing-ffp2-mask.jpg?s=2048x2048&w=is&k=20&c=-9JbJFpmyQKWbif_P23mZX6dll0lo4F8fBuU-SzTzqY=
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
Predicted class: with_mask with confidence 1.00


In [7]:

# Example usage
image_url = input("Enter image URL: ")
test_image(image_url, model, list(train_dataset.class_indices.keys()))

Enter image URL: https://media.istockphoto.com/id/1443562748/photo/cute-ginger-cat.jpg?s=2048x2048&w=is&k=20&c=266RJoIIQDUVv1JVMaxI4mUcUX_2hzV_W_7S5IQqnBw=
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Predicted class: with_mask with confidence 0.99
